In [12]:
import pandas as pd

def load_final(fname, package):
    df = pd.read_csv(fname)
    df['package'] = package

    # 1) fix any weird R2 >1
    if df.loc[0,'test_r2'] > 1:
        df['test_r2'] = df['test_r2'] / 100.0

    # 2) rename CatBoost iterations → n_estimators
    if 'iterations' in df.columns:
        df = df.rename(columns={'iterations':'n_estimators'})

    # 3) drop or rename rsm
    if 'rsm' in df.columns:
        df = df.drop(columns=['rsm'])

    # 4) unify colsample
    df = df.rename(columns={
        'colsample_bytree':'colsample',
        'colsample_bylevel':'colsample',
    })

    # 5) unify seed/random_seed → random_state
    if 'seed' in df.columns:
        df = df.rename(columns={'seed':'random_state'})
    if 'random_seed' in df.columns:
        df = df.rename(columns={'random_seed':'random_state'})

    # 6) rename CatBoost depth → max_depth
    if 'depth' in df.columns:
        df = df.rename(columns={'depth':'max_depth'})

    # 7) hard‐code XGBoost’s lr
    if package == "XGBoost":
        df['learning_rate'] = 0.09877646890750276

    # select + order
    cols = [
      'package','random_state','learning_rate','n_estimators','max_depth',
      'subsample','colsample','feature_subsample',
      'n_layers','units','dropout',
      'train_time_s','pred_time_s',
      'test_rmse','test_r2'
    ]
    return df[[c for c in cols if c in df.columns]]

# load each
lgbm = load_final("rq1_lgbm_final.csv", "LightGBM")
xgb  = load_final("rq1_xgb_final.csv",  "XGBoost")
cb   = load_final("rq1_cb_final.csv",   "CatBoost")
mlp  = load_final("rq1_mlp_final.csv",  "MLP")

# concat and write
df_all = pd.concat([lgbm, xgb, cb, mlp], ignore_index=True)
df_all.to_csv("rq1_package_comparison.csv", index=False)
df_all

,package,random_state,learning_rate,n_estimators,max_depth,subsample,colsample,train_time_s,pred_time_s,test_rmse,test_r2,n_layers,units,dropout
0,LightGBM,7.0,0.088844,1000.0,12.0,0.660001,0.996979,3.211,0.393,6878.613,0.9158,NaN,NaN,NaN
1,XGBoost,999.0,0.098776,356.0,5.0,0.993475,0.576062,3.288,0.032,6449.413,0.9260,NaN,NaN,NaN
2,CatBoost,123.0,0.100000,1000.0,7.0,0.500000,0.500000,10.611,1.189,5452.723,0.9471,NaN,NaN,NaN
3,MLP,NaN,0.009873,NaN,NaN,NaN,NaN,39.770,0.586,6359.375,0.9281,3.0,178.0,0.12973
